# NEURO AI LAB · Swin-Tiny V3 fine-tuning

Colab'da GPU çalışma zamanı seçin ve hücreleri sırayla çalıştırın. Drive bağlama istemini kendi hesabınızla onaylayın. Her çalıştırma kendine ait çıktı klasörlerine yazılır.

Drive girdi klasörü: `MyDrive/Beyin_Tumoru_Projesi`. Gerekenler: `Brain_Tumor_MRI_Dataset.zip`, `veri_bolunmesi_v3/` ve `modeller/swin_tiny_v3_best.pth`.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import subprocess
import sys

import torch
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError("Colab çalışma zamanı türünden GPU seçin.")
print("GPU:", torch.cuda.get_device_name(0))
drive.mount("/content/drive")

project = Path("/content/drive/MyDrive/Beyin_Tumoru_Projesi")
archive = project / "Brain_Tumor_MRI_Dataset.zip"
splits = project / "veri_bolunmesi_v3"
base = project / "modeller/swin_tiny_v3_best.pth"
for path in (archive, splits / "train.csv", splits / "val.csv", base):
    if not path.is_file():
        raise FileNotFoundError(f"Eksik girdi: {path}")

repo = Path("/content/neuro-ai-lab-swin-v3")
if not repo.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", "main",
        "https://github.com/saglamkubra/Brain-Tumor-MRI-Classification-Segmentation.git",
        str(repo),
    ], check=True)
for script in ("train.py", "export_results.py"):
    if not (repo / "experiments/swin_v3" / script).is_file():
        raise FileNotFoundError(f"GitHub kopyasında betik eksik: {script}")
revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "timm==1.0.22",
    "scikit-learn", "matplotlib", "Pillow",
], check=True)

In [ ]:
run = datetime.now(timezone.utc).strftime("colab_%Y%m%dT%H%M%S%fZ")
weights = project / "modeller" / "swin_v3_runs" / run
results = project / "results" / "swin_v3" / "finetuning" / run
weights.mkdir(parents=True, exist_ok=False)
results.mkdir(parents=True, exist_ok=False)
(results / "source_revision.txt").write_text(revision, encoding="utf-8")

subprocess.run([
    sys.executable, "-u", "-m", "experiments.swin_v3.train",
    "--archive", str(archive), "--splits", str(splits), "--base", str(base),
    "--output", str(weights), "--results", str(results),
], cwd=repo, check=True)
subprocess.run([
    sys.executable, "-m", "experiments.swin_v3.export_results", str(results),
], cwd=repo, check=True)
print("Yeni checkpoint:", weights / "swin_tiny_v3_finetuned_best.pth")
print("Metrikler ve grafikler:", results)

In [ ]:
import json

metrics = json.loads((results / "metrics.json").read_text(encoding="utf-8"))
test = metrics["test"]
print("En iyi epoch:", metrics["metadata"]["epoch"])
print("Test görüntüsü:", test["n"])
print(f"Accuracy: {test['accuracy']:.2%}")
print(f"Macro F1: {test['classification']['macro avg']['f1-score']:.2%}")
print(f"Glioma recall: {test['classification']['glioma']['recall']:.2%}")
print("Glioma → notumor:", test["glioma_to_notumor"])